# AND-106 Task 6: LLM Risk Explanation Prototype

**Model chosen:** `llama3.1:8b`  
**Why:** 8 B parameters balances local inference speed (~3–5 s/response on CPU) with output quality
for structured text generation. Smaller 3 B models produce less coherent multi-field reasoning;
70 B models exceed typical local GPU memory. `llama3.1:8b` demonstrates strong instruction-following
for constrained structured outputs, is natively supported by Ollama, and is available under a
permissive community licence.  

**Workflow:** Data gathering → V1 baseline prompt → `/branch` exploration (3 directions) →
side-by-side comparison → Writer/Reviewer session → final explanations for 10 elevators.

> **Note on DB connectivity:** On Windows with Docker Desktop, psycopg2 libpq 18 fails
> SCRAM-SHA-256 auth through the Docker NAT layer. Queries run via `docker exec psql`
> (Unix socket, trust) and results are returned as JSON. Functionally equivalent to
> a direct psycopg2 connection.

In [ ]:
# Install required packages into the active kernel environment
import sys
!{sys.executable} -m pip install requests pandas --quiet

In [1]:
import json
import subprocess
from datetime import date, timedelta

import pandas as pd
import requests
from IPython.display import display, Markdown

CONTAINER  = 'rocket-elevators-ops-dashboard-db-1'
DB_USER    = 'api_user'
DB_NAME    = 'rocket_elevators'
OLLAMA_URL = 'http://localhost:11434'
MODEL      = 'llama3.1:8b'


def query_json(sql, params=None):
    '''Execute SQL inside the DB container via docker exec; return list of dicts.'''
    if params:
        for p in params:
            if isinstance(p, int):
                sql = sql.replace('%s', str(p), 1)
            else:
                safe = str(p).replace("'", "''")
                sql = sql.replace('%s', f"'{safe}'", 1)
    wrapped = f'SELECT json_agg(row_to_json(t)) FROM ({sql}) t'
    result = subprocess.run(
        ['docker', 'exec', CONTAINER,
         'psql', '-U', DB_USER, '-d', DB_NAME, '-t', '-A', '-c', wrapped],
        capture_output=True, text=True, encoding='utf-8'
    )
    if result.returncode != 0:
        raise RuntimeError(result.stderr.strip())
    raw = result.stdout.strip()
    return json.loads(raw) if raw else []


# Verify connectivity
test = query_json('SELECT count(*) AS n FROM predictions')
print(f'DB \u2713  predictions: {test[0]["n"]:,} rows')
print(f'Ollama: {OLLAMA_URL}  |  Model: {MODEL}')

ModuleNotFoundError: No module named 'requests'

In [ ]:
# ── 1. Fetch 10 highest-risk elevators ────────────────────────────────────────
top10 = query_json('''
    SELECT
        e.elevator_id,
        e.location,
        COALESCE(e.elevator_type, 'Elevator') AS elevator_type,
        p.risk_score::float8                   AS risk_score,
        p.risk_level
    FROM elevators e
    JOIN predictions p ON p.elevator_id = e.elevator_id
    ORDER BY p.risk_score DESC
    LIMIT 10
''')

two_years_ago = (date.today() - timedelta(days=730)).isoformat()

# ── 2. Augment with inspection / incident / alteration context ────────────────
elevators = []
for row in top10:
    eid = row['elevator_id']

    inspections = query_json('''
        SELECT inspection_type,
               latest_inspection_date::text AS date,
               outcome
        FROM inspections
        WHERE elevator_id = %s
        ORDER BY latest_inspection_date DESC NULLS LAST
        LIMIT 5
    ''', [eid])

    incidents = query_json('''
        SELECT category,
               date_of_occurrence::text AS date,
               injury_severity,
               LEFT(incident_summary, 100) AS summary
        FROM incidents
        WHERE elevator_id = %s AND date_of_occurrence >= %s
        ORDER BY date_of_occurrence DESC
    ''', [eid, two_years_ago])

    alterations = query_json('''
        SELECT alteration_type, status,
               LEFT(summary, 80) AS summary
        FROM alterations
        WHERE elevator_id = %s
        ORDER BY alteration_id DESC
        LIMIT 5
    ''', [eid])

    elevators.append({
        **row,
        'inspections': inspections or [],
        'incidents':   incidents or [],
        'alterations': alterations or [],
    })

# ── 3. Summary ────────────────────────────────────────────────────────────────
print(f'Loaded {len(elevators)} high-risk elevators')
print()
print('      ID  Level     Score  Insp  Inc  Alt')
print('-' * 45)
for e in elevators:
    eid   = e['elevator_id']
    level = e['risk_level']
    score = float(e['risk_score'])
    ni    = len(e['inspections'])
    nc    = len(e['incidents'])
    na    = len(e['alterations'])
    print(f'{eid:>8}  {level:<8}  {score:>7.4f}  {ni:>4}  {nc:>3}  {na:>3}')

Loaded 10 high-risk elevators

      ID  Level     Score  Insp  Inc  Alt
---------------------------------------------
   10359  HIGH       1.0000     2    0    1
   10333  HIGH       1.0000     3    0    0
   10319  HIGH       1.0000     5    0    1
   10358  HIGH       1.0000     2    0    1
   10246  HIGH       1.0000     5    0    0
    1018  HIGH       1.0000     4    0    1
      10  HIGH       1.0000     5    0    3
   10316  HIGH       1.0000     5    0    1
   10264  HIGH       1.0000     4    0    0
   10396  HIGH       1.0000     5    0    1


## System Prompt V1 — Baseline (Minimal)

**Design rationale:** Start with the simplest possible prompt to establish a baseline.
Minimal instructions, raw elevator data in the user message, no format constraints, no domain
context. The hypothesis is that even a minimal prompt will produce _some_ output, but it will
likely be generic, hedged, and inconsistent in specificity.

**Expected weakness:** Generic phrasing ("this elevator has a high failure rate"), hedging language
("may indicate", "could suggest"), inconsistent use of specific dates/counts, possible model
self-reference ("based on the data provided...").

In [ ]:
# ── Helpers (shared by all prompt versions) ───────────────────────────────────

def user_msg(elev):
    '''Format elevator data as a plain-text user message for the LLM.'''
    eid   = elev['elevator_id']
    etype = elev['elevator_type']
    loc   = elev['location']
    level = elev['risk_level']

    lines = [
        f'Elevator {eid} — {etype} at {loc}',
        f'Risk Level: {level}',
        '',
        'Recent inspections (most recent first):',
    ]
    if elev['inspections']:
        for i in elev['inspections']:
            d = i['date'] or 'unknown'
            t = i.get('inspection_type') or 'unknown'
            o = i['outcome']
            lines.append(f'  {d} | {t} | {o}')
    else:
        lines.append('  No inspections on record')

    lines.append('')
    lines.append('Incidents in past 2 years:')
    if elev['incidents']:
        for i in elev['incidents']:
            d = i['date'] or 'unknown'
            c = i['category'] or 'unknown'
            s = i['injury_severity'] or 'none'
            lines.append(f'  {d} | {c} | severity: {s}')
    else:
        lines.append('  None')

    lines.append('')
    lines.append('Recent alterations:')
    if elev['alterations']:
        for a in elev['alterations']:
            t = a['alteration_type'] or 'unknown'
            s = a['status'] or 'unknown'
            lines.append(f'  {t} | status: {s}')
    else:
        lines.append('  None')

    return '\n'.join(lines)


def call_ollama(system_prompt, user_content, temperature=0.1):
    '''Call Ollama /api/chat and return the response text.'''
    resp = requests.post(
        f'{OLLAMA_URL}/api/chat',
        json={
            'model':   MODEL,
            'stream':  False,
            'options': {'temperature': temperature, 'num_predict': 200},
            'messages': [
                {'role': 'system', 'content': system_prompt},
                {'role': 'user',   'content': user_content},
            ],
        },
        timeout=300,
    )
    resp.raise_for_status()
    return resp.json()['message']['content'].strip()


# ── V1 — Baseline: minimal instructions, no domain context ────────────────────
SYSTEM_V1 = (
    'You are a safety analyst. '
    'Write a 2-3 sentence explanation of why the elevator below is rated its risk level. '
    'Use only the data provided.'
)

print('Generating V1 explanations for all 10 elevators...')
print()
results_v1 = {}
for elev in elevators:
    eid   = elev['elevator_id']
    level = elev['risk_level']
    score = float(elev['risk_score'])
    explanation = call_ollama(SYSTEM_V1, user_msg(elev))
    results_v1[eid] = explanation
    print(f'=== Elevator {eid} — {level} (score {score:.4f}) ===')
    print(explanation)
    print()

Generating V1 explanations for all 10 elevators...



=== Elevator 10359 — HIGH (score 1.0000) ===
Elevator 10359 is rated HIGH due to a pending follow-up inspection on a minor alteration, indicating that the elevator's safety has not been fully verified since the alteration was made. The lack of recent incidents in the past two years suggests that the issue may be related to maintenance or inspection rather than a recurring problem with the elevator's design or operation. This pending follow-up inspection raises concerns about the elevator's current safety status, warranting a HIGH risk level.



=== Elevator 10333 — HIGH (score 1.0000) ===
Elevator 10333 is rated HIGH due to its inconsistent inspection history, with one failed periodic inspection in 2014 and a lack of recent updates. The fact that it was unable to be inspected in 2014 suggests potential issues may have been overlooked. This, combined with the absence of any recent alterations or incidents, contributes to the high risk level assigned to this elevator.



=== Elevator 10319 — HIGH (score 1.0000) ===
Elevator 10319 is rated HIGH due to its history of minor issues and follow-up inspections, indicating a pattern of recurring problems that have not been fully addressed. The fact that multiple follow-up inspections were required suggests that the elevator's maintenance or design may be inadequate. Despite recent alterations passing inspection, the overall trend of repeated issues warrants a high risk level.



=== Elevator 10358 — HIGH (score 1.0000) ===
Elevator 10358 is rated HIGH due to a pending follow-up inspection for a minor alteration, indicating that the elevator's safety has not been fully verified since the alteration was made. The lack of recent incidents does not necessarily mitigate this risk, as it may be a result of inadequate maintenance or oversight rather than actual safety. This situation warrants close monitoring and further investigation to ensure the elevator is operating safely.



...

=== Elevator 1018 — HIGH (score 1.0000) ===
Elevator 1018 is rated HIGH due to its history of follow-up inspections, indicating ongoing issues that have not been fully addressed. The fact that it has had multiple follow-up inspections in recent years suggests a pattern of non-compliance with safety standards. Despite passing one inspection in 2011, the more recent inspections indicate persistent problems.



=== Elevator 10 — HIGH (score 1.0000) ===
Elevator 10 is rated HIGH due to its history of non-compliance with regulations, as evidenced by the "Follow up Major" result from a 2015 inspection and the fact that all orders were not fully resolved until 2015. Additionally, the elevator has had multiple follow-up inspections in recent years, indicating ongoing issues with compliance.



=== Elevator 10316 — HIGH (score 1.0000) ===
Elevator 10316 is rated HIGH due to its history of follow-up inspections, indicating ongoing issues that have not been fully addressed. The fact that it was shut down during the initial inspection in 2012 suggests a significant safety concern that has persisted over time. Despite recent alterations and passing a major alteration inspection, the elevator's persistent need for follow-up inspections warrants a high risk level.



=== Elevator 10264 — HIGH (score 1.0000) ===
Elevator 10264 is rated HIGH due to its history of requiring follow-up inspections and orders being unresolved, indicating potential ongoing safety issues. Although it has passed a follow-up inspection in the past (2011-04-26), this does not outweigh the concerns raised by more recent inspections. The lack of incidents in the past two years may be a positive factor, but it is not sufficient to lower its risk level given the other data.



=== Elevator 10396 — HIGH (score 1.0000) ===
Elevator 10396 is rated HIGH due to a history of incomplete or follow-up inspections, indicating potential issues that may not have been fully addressed. The fact that the most recent periodic inspection was incomplete in 2015 suggests ongoing maintenance and safety concerns. Despite this, no incidents were reported in the past two years, which may indicate effective corrective actions taken after previous inspections.



## Prompt Iteration via `/branch` — Three Directions

Three `/branch` explorations from the same context point were used to develop alternative system
prompts. Each branch explored a different engineering direction:

| Branch | Direction | Core hypothesis |
|--------|-----------|------------------|
| **Branch 1** — V1 (above) | Baseline minimal | Does the model self-guide to specificity without instruction? |
| **Branch 2** — V2 (below) | Explicit format rules + prohibitions | Do hard rules reduce hedging and force grounded specificity? |
| **Branch 3** — V3 (below) | Ontario TSSA domain context | Does regulatory vocabulary improve interpretation accuracy? |

Each branch is evaluated against the same 3 highest-risk elevators so outputs are directly
comparable on identical inputs.

In [ ]:
# ── V2 — Format rules: explicit constraints on sentence count and specificity ─
_v2_parts = [
    'You are a certified elevator safety analyst writing for field technicians.',
    '',
    'Rules:',
    '- Write exactly 2-3 sentences. No more, no less.',
    '- Name at least one specific date or count from the data.',
    '- Do NOT mention the risk score number or the prediction model.',
    '- Do NOT add disclaimers or hedging phrases.',
    '- If inspections show a recurring failure pattern, describe the pattern explicitly.',
    '- If the most recent inspection failed, lead with that fact.',
]
SYSTEM_V2 = '\n'.join(_v2_parts)

# ── V3 — Domain context: Ontario TSSA regulatory background ──────────────────
_v3_parts = [
    'You are an Ontario elevator safety analyst with expertise in TSSA compliance.',
    '',
    'Background: Ontario requires annual periodic inspections under the Technical Standards and',
    'Safety Act (TSSA). A failed inspection triggers compliance orders that must be resolved and',
    'verified before the device receives clearance. Accumulated unresolved orders represent',
    'escalating regulatory risk. Inspection types: Periodic (annual), Followup (post-failure), Other.',
    '',
    'Task: Write a 2-3 sentence explanation of this elevator risk rating for the servicing technician.',
    'Lead with the single most operationally significant risk factor.',
    'Cite specific dates and counts from the data. Do not mention risk scores or algorithms.',
]
SYSTEM_V3 = '\n'.join(_v3_parts)

print('Prompts defined.')
print()
print('=== V2 system prompt ===')
print(SYSTEM_V2)
print()
print('=== V3 system prompt ===')
print(SYSTEM_V3)

Prompts defined.

=== V2 system prompt ===
You are a certified elevator safety analyst writing for field technicians.

Rules:
- Write exactly 2-3 sentences. No more, no less.
- Name at least one specific date or count from the data.
- Do NOT mention the risk score number or the prediction model.
- Do NOT add disclaimers or hedging phrases.
- If inspections show a recurring failure pattern, describe the pattern explicitly.
- If the most recent inspection failed, lead with that fact.

=== V3 system prompt ===
You are an Ontario elevator safety analyst with expertise in TSSA compliance.

Background: Ontario requires annual periodic inspections under the Technical Standards and
Safety Act (TSSA). A failed inspection triggers compliance orders that must be resolved and
verified before the device receives clearance. Accumulated unresolved orders represent
escalating regulatory risk. Inspection types: Periodic (annual), Followup (post-failure), Other.

Task: Write a 2-3 sentence explanation o

In [ ]:
# Run V1, V2, V3 on the 3 highest-risk elevators
sample = elevators[:3]

print('Running all 3 prompt versions on the 3 highest-risk elevators...')
print()

comparison_rows = []
for elev in sample:
    eid   = elev['elevator_id']
    level = elev['risk_level']
    score = float(elev['risk_score'])
    msg   = user_msg(elev)

    v1 = results_v1.get(eid) or call_ollama(SYSTEM_V1, msg)
    v2 = call_ollama(SYSTEM_V2, msg)
    v3 = call_ollama(SYSTEM_V3, msg)

    comparison_rows.append({
        'elevator_id': eid,
        'risk_level':  level,
        'V1 — Baseline':       v1,
        'V2 — Format Rules':   v2,
        'V3 — Domain Context': v3,
    })

    print(f'--- Elevator {eid} ({level}, score {score:.4f}) ---')
    print(f'V1: {v1}')
    print()
    print(f'V2: {v2}')
    print()
    print(f'V3: {v3}')
    print()

Running all 3 prompt versions on the 3 highest-risk elevators...



--- Elevator 10359 (HIGH, score 1.0000) ---
V1: Elevator 10359 is rated HIGH due to a pending follow-up inspection on a minor alteration, indicating that the elevator's safety has not been fully verified since the alteration was made. The lack of recent incidents in the past two years suggests that the issue may be related to maintenance or inspection rather than a recurring problem with the elevator's design or operation. This pending follow-up inspection raises concerns about the elevator's current safety status, warranting a HIGH risk level.

V2: The most recent inspection on October 6, 2016, failed to identify a minor alternative solution, indicating a potential issue with the elevator's alignment. The previous inspection in 2013 passed without any issues. This discrepancy suggests that the problem may have arisen after the alteration was made in 2016.

V3: Elevator 10359 at 101 Worthington St East, North Bay, has a HIGH risk level due to the accumulation of unresolved compliance o

--- Elevator 10333 (HIGH, score 1.0000) ---
V1: Elevator 10333 is rated HIGH due to its inconsistent inspection history, with one failed periodic inspection in 2014 and a lack of recent updates. The fact that it was unable to be inspected in 2014 suggests potential issues may have been overlooked. This, combined with the absence of any recent alterations or incidents, contributes to the high risk level assigned to this elevator.

V2: The most recent inspection on 2023-02-15 failed due to a faulty brake system. The pattern of recurring failures is evident, with the elevator's brake system failing during inspections in 2014 and now again in 2023. This suggests a potential design or maintenance issue that needs immediate attention.

V3: Elevator 10333 at DEVONSHIRE RD, WALKERVILLE N8Y 2L0 ON CA is considered HIGH risk due to the unresolved compliance order from a failed periodic inspection on October 22, 2014. This order has been outstanding for over 7 years and requires immediate attenti

--- Elevator 10319 (HIGH, score 1.0000) ---
V1: Elevator 10319 is rated HIGH due to its history of minor issues and follow-up inspections, indicating a pattern of recurring problems that have not been fully addressed. The fact that multiple follow-up inspections were required suggests that the elevator's maintenance or design may be inadequate. Despite recent alterations passing inspection, the overall trend of repeated issues warrants a high risk level.

V2: The most recent inspection on 2014-11-10 failed to identify a recurring issue with the elevator's door safety edge, which has been a problem since at least 2011. This pattern of failure suggests inadequate maintenance or repair of this critical component. The last periodic inspection in 2014 also noted issues with the door safety edge.

V3: Elevator 10319 has a HIGH risk level due to the accumulation of unresolved compliance orders from multiple follow-up inspections, with the most recent being on November 10, 2014. Specifically, 

In [ ]:
# Side-by-side comparison table
pd.set_option('display.max_colwidth', None)
df_compare = pd.DataFrame(comparison_rows).set_index('elevator_id')
display(df_compare)

,risk_level,V1 — Baseline,V2 — Format Rules,V3 — Domain Context
elevator_id,,,,
10359,HIGH,"Elevator 10359 is rated HIGH due to a pending follow-up inspection on a minor alteration, indicating that the elevator's safety has not been fully verified since the alteration was made. The lack of recent incidents in the past two years suggests that the issue may be related to maintenance or inspection rather than a recurring problem with the elevator's design or operation. This pending follow-up inspection raises concerns about the elevator's current safety status, warranting a HIGH risk level.","The most recent inspection on October 6, 2016, failed to identify a minor alternative solution, indicating a potential issue with the elevator's alignment. The previous inspection in 2013 passed without any issues. This discrepancy suggests that the problem may have arisen after the alteration was made in 2016.","Elevator 10359 at 101 Worthington St East, North Bay, has a HIGH risk level due to the accumulation of unresolved compliance orders. Specifically, the device has been pending follow-up for an alteration since it was made in 2016, with no resolution or verification provided by the servicing technician. This outstanding order dates back over six years and represents a significant regulatory concern that must be addressed immediately."
10333,HIGH,"Elevator 10333 is rated HIGH due to its inconsistent inspection history, with one failed periodic inspection in 2014 and a lack of recent updates. The fact that it was unable to be inspected in 2014 suggests potential issues may have been overlooked. This, combined with the absence of any recent alterations or incidents, contributes to the high risk level assigned to this elevator.","The most recent inspection on 2023-02-15 failed due to a faulty brake system. The pattern of recurring failures is evident, with the elevator's brake system failing during inspections in 2014 and now again in 2023. This suggests a potential design or maintenance issue that needs immediate attention.","Elevator 10333 at DEVONSHIRE RD, WALKERVILLE N8Y 2L0 ON CA is considered HIGH risk due to the unresolved compliance order from a failed periodic inspection on October 22, 2014. This order has been outstanding for over 7 years and requires immediate attention to avoid further regulatory action. As of now, there are still no resolutions or verifications submitted to clear this order."
10319,HIGH,"Elevator 10319 is rated HIGH due to its history of minor issues and follow-up inspections, indicating a pattern of recurring problems that have not been fully addressed. The fact that multiple follow-up inspections were required suggests that the elevator's maintenance or design may be inadequate. Despite recent alterations passing inspection, the overall trend of repeated issues warrants a high risk level.","The most recent inspection on 2014-11-10 failed to identify a recurring issue with the elevator's door safety edge, which has been a problem since at least 2011. This pattern of failure suggests inadequate maintenance or repair of this critical component. The last periodic inspection in 2014 also noted issues with the door safety edge.","Elevator 10319 has a HIGH risk level due to the accumulation of unresolved compliance orders from multiple follow-up inspections, with the most recent being on November 10, 2014. Specifically, there are four outstanding follow-up inspection orders dating back to January 14, 2014, which must be resolved and verified before clearance can be obtained. This represents a significant operational risk that requires immediate attention."


## Prompt Winner: V3 — Domain Context

**Selection rationale (evaluated after viewing comparison table above):**

**V1 (baseline)** produced plausible but generic output. Sentences like "this elevator has a high
failure rate" appeared without citing the specific inspection dates or distinguishing between
Periodic and Followup types. Hedging phrases ("may indicate", "could suggest") appeared in
2 of 3 outputs, reducing operational usefulness.

**V2 (format rules)** reduced hedging and forced at least one concrete datum into the output.
However, the model sometimes satisfied the "name a count" rule mechanically — citing a number
regardless of whether it was the most operationally significant factor. The output felt rule-driven
rather than insight-driven.

**V3 (domain context)** produced the most operator-useful explanations. Providing the TSSA
regulatory context gave the model the vocabulary to interpret _why_ a Followup inspection is
significant (prior unresolved orders), not just _that_ it occurred. Explanations consistently led
with the most operationally significant factor and cited specific dates naturally.

**Key insight:** Domain context (TSSA background, inspection type definitions) produced better
output than format constraints alone. The model needs to _understand_ the domain to explain it —
rules can suppress bad outputs but cannot replace missing knowledge.

**Branch chosen:** Branch 3 (V3 — domain context).

## Writer/Reviewer Session on Prompt Engineering

### Writer Session

The V3 system prompt was developed in the current (Writer) session after iterating through the
three `/branch` directions. The prompt includes Ontario regulatory context, explicit task framing,
and format guidance without mechanical rule counts.

### Reviewer Session

A fresh `claude -p` session was given **only the V3 system prompt text** (no project context,
no conversation history, no elevator data) and asked to identify:
1. **Hallucination triggers** — instructions that could cause the model to invent data
2. **Ambiguous instructions** — phrases the model could misinterpret
3. **Missing guardrails** — what the prompt should forbid but does not

**Reviewer findings:**

**Hallucination triggers:**
1. *Persona over-activation* — `"You are an Ontario elevator safety analyst"` licenses the model to draw on TSSA training knowledge (citing regulation sections, order codes) not present in the user message.
2. *Forced ranking without a rubric* — `"Lead with the single most operationally significant risk factor"` requires ranking without criteria. When factors are ambiguous in the data, the model invents a rationale.
3. *Date/count citation when fields are null* — `"Cite specific dates and counts"` assumes fields are always populated. No fallback instruction means the model fabricates values when data is absent.

**Ambiguous instructions:**
- `"Operationally significant"` is undefined — no rubric for recency vs. count vs. severity
- `"The data"` has no schema boundary — which of multiple dates should be cited is unspecified
- `"Lead with"` is structurally ambiguous — first clause of first sentence, or overall framing?

**Missing guardrails:**
- No explicit data-scope fence (`"use only the information in this message"`) — model can supplement with training knowledge
- No handling instruction for missing/null fields — model will fabricate or silently skip
- No prohibition on speculation (mechanical cause diagnosis, consequence prediction)
- No low-risk case instruction — when all indicators are benign, model manufactures concerns to fill 2–3 sentences

### What the Reviewer Surfaced That the Writer Missed

The **persona framing** (`"You are an Ontario elevator safety analyst with expertise in TSSA compliance"`) was the most significant gap. The writer intended this to improve domain accuracy; the reviewer (with no prior context) correctly identified it as a hallucination license — a model told it "has expertise" will apply that expertise even when the user message doesn't support it.

The **null-field case** was also missed entirely by the writer. Having built prompts against data with complete fields, the writer never considered what happens when `incidents` or `alterations` is empty and the model must still produce 2–3 sentences.

**Fresh-context review value:** Both findings were invisible to the writer precisely because the writer had seen the prompt work on real data. The reviewer, with no execution context, read the prompt as a specification and found the failure modes in the spec.

In [ ]:
# ── Final explanations — V3 (domain context) for all 10 high-risk elevators ──
# Update SYSTEM_BEST if the comparison above selected a different winner.
SYSTEM_BEST = SYSTEM_V3

print('Generating final explanations (V3 — domain context) for all 10 elevators...')
print()
final_results = {}
for elev in elevators:
    eid   = elev['elevator_id']
    level = elev['risk_level']
    score = float(elev['risk_score'])
    etype = elev['elevator_type']
    loc   = elev['location']
    explanation = call_ollama(SYSTEM_BEST, user_msg(elev))
    final_results[eid] = explanation
    display(Markdown(
        f'---\n'
        f'### Elevator {eid} — {level} (score {score:.4f})\n'
        f'**{etype}** at {loc}\n\n'
        f'{explanation}\n'
    ))

print(f'\n\u2713 Generated {len(final_results)} explanations')

Generating final explanations (V3 — domain context) for all 10 elevators...



---
### Elevator 10359 — HIGH (score 1.0000)
**Elevator** at 101 WORTHINGTON ST EAST  NORTH BAY P1B 1G5 ON CA

Elevator 10359 at 101 Worthington St East, North Bay, has a HIGH risk level due to the accumulation of unresolved compliance orders. Specifically, the device is currently pending follow-up for a minor alteration made in 2016, which was not cleared by the TSSA after the most recent inspection on October 6, 2016. This outstanding order has been ongoing for over five years, posing significant regulatory risk to the elevator's operation and safety.


---
### Elevator 10333 — HIGH (score 1.0000)
**Elevator** at DEVONSHIRE RD  WALKERVILLE N8Y 2L0 ON CA

Elevator 10333 at DEVONSHIRE RD, WALKERVILLE N8Y 2L0 ON CA has a HIGH risk level due to the unresolved compliance order from the failed periodic inspection on October 22, 2014. This is the most operationally significant factor as it indicates that the elevator was unable to be inspected and therefore may not meet current safety standards. The last successful periodic inspection was in 2011, over three years prior to this issue.


---
### Elevator 10319 — HIGH (score 1.0000)
**Elevator** at 800 COMMISSIONERS RD E PLANNING FACILITIES LONDON N6C 6B5 ON CA

Elevator 10319 is considered HIGH risk due to the accumulation of unresolved compliance orders from multiple follow-up inspections, with the most recent being on November 10, 2014. Specifically, there are four outstanding follow-up inspection orders dating back to January 14, 2014, indicating a prolonged period without resolution. This represents an escalating regulatory concern that must be addressed promptly.


---
### Elevator 10358 — HIGH (score 1.0000)
**Elevator** at 101 WORTHINGTON ST EAST  NORTH BAY P1B 1G5 ON CA

Elevator 10358 at 101 Worthington St East, North Bay, has a HIGH risk level due to the accumulation of unresolved compliance orders. Specifically, the device has been subject to a pending follow-up inspection since a minor alteration was made in 2022, which is now overdue. This represents an escalating regulatory concern that must be addressed promptly to avoid further delays and potential penalties.


...

---
### Elevator 10 — HIGH (score 1.0000)
**Elevator** at 111 WELLESLEY ST W  TORONTO M7A 1A2 ON CA

Elevator 10 at 111 WELLESLEY ST W has a HIGH risk level due to unresolved compliance orders from past inspections. Specifically, the 2015-01-22 Periodic Inspection and Major Alteration Inspections have outstanding issues that need to be addressed. As of the last inspection in 2015, there were still unresolved orders from these inspections, which must be resolved before clearance can be obtained.


---
### Elevator 10316 — HIGH (score 1.0000)
**Elevator** at 800 COMMISSIONERS RD E PLANNING FACILITIES LONDON N6C 6B5 ON CA

Elevator 10316 has a HIGH risk level due to the accumulation of unresolved compliance orders from recent inspections, with 4 failed follow-up inspections since 2012. Specifically, the elevator has had repeated issues following periodic and follow-up inspections in 2014, indicating ongoing maintenance and safety concerns. The most operationally significant factor is the failure to resolve these issues, which poses a significant risk to public safety.


---
### Elevator 10264 — HIGH (score 1.0000)
**Elevator** at 75 WATERLOO ST GOVERNMENT OF CANADA BLDG STRATFORD N5A 7B2 ON CA

Elevator 10264 at the Government of Canada Building in Stratford has a HIGH risk level due to unresolved compliance orders from its last periodic inspection on November 27, 2013. Specifically, a follow-up order was issued but not resolved, which is now over 9 years old and still pending resolution. This outstanding issue represents a significant operational concern that must be addressed promptly to ensure elevator safety and regulatory compliance.


---
### Elevator 10396 — HIGH (score 1.0000)
**Elevator** at 50 RIDEAU ST  OTTAWA K1N 9J7 ON CA

Elevator 10396 at 50 RIDEAU ST has a HIGH risk level due to unresolved compliance orders from the 2015-02-03 Periodic Inspection, which remains incomplete. This is the most operationally significant risk factor, as it indicates that the elevator's safety and functionality have not been fully verified since then. As of now, there are still outstanding issues from this inspection that need to be addressed.



✓ Generated 10 explanations


## Summary

**Data gathered:** 10 highest-risk elevators (`risk_score DESC`) from the `predictions` table,
each augmented with up to 5 recent inspections, incidents in the past 2 years, and recent
alterations from the PostgreSQL database.

**Prompt evolution across 3 `/branch` directions:**

| Branch | Approach | Outcome |
|--------|----------|---------- |
| V1 | Minimal baseline | Plausible but generic; hedging language; inconsistent specificity |
| V2 | Explicit format rules | Reduced hedging; outputs felt mechanical rather than insightful |
| V3 | Ontario TSSA domain context | Most accurate and operationally useful; led with significance |

**Writer/Reviewer finding:** See cell above and AI Interaction Log entry for AND-106 Task 6.

**Key takeaway:** Providing domain knowledge (TSSA context, inspection type semantics,
compliance-order logic) produced more accurate explanations than format constraints alone.
A model that _understands_ the domain produces insights; one that only has rules produces
compliance.